# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [70]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv
from openai import AuthenticationError, APIConnectionError
from pydantic import ValidationError
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum
import json
import time
from unittest import result
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import gradio as gr
import time as pytime


load_dotenv(".env")

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = "provide your email"
EMAIL_PASSWORD = "password"

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [71]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Pydantic Models for Validation

In [72]:
class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [73]:
def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [74]:
def parse_json_response(content: str) -> dict:
    """Clean Markdown code fences and parse JSON content."""

    if "```json" in content:
        content = content.split("```json")[1].split("```")[0]
    elif "```" in content:
        content = content.split("```")[1].split("```")[0]

    return json.loads(content)

def validate_meal_plan(data: dict) -> MealPlan:
    """Validate meal plan data using the Pydantic model."""
    return MealPlan(**data)

class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.shopping_parser = JsonOutputParser(pydantic_object=ShoppingItem)

    def _request_meal_plan(
        self,
        dietary_preference: str,
        theme_requests: str,
        additional_preferences: str
    ):
        """Request a meal plan from the LLM."""

        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        return recipe_chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": theme_requests if theme_requests else "random diverse cuisines",
            "additional_preferences": additional_preferences if additional_preferences else "none",
            "format_instructions": self.recipe_parser.get_format_instructions()
        })

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        try:
            result = self._request_meal_plan(
                dietary_preference,
                theme_requests,
                additional_preferences
            )
            
            meal_plan_data = parse_json_response(result.content)
            meal_plan = validate_meal_plan(meal_plan_data)
            generation_time = time.time() - start_time

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(generation_time, 2),
                "error": None
            }

        except AuthenticationError as e:
            return {
                "success": False,
                "meal_plan": None,
                "error": (
                    f"Function: MealPlanGenerator._request_meal_plan | "
                    f"Error type: {type(e).__name__} | "
                    "Location: OpenAI API authentication | "
                    "Suggestion: Check your OPENAI_API_KEY in the .env file."
                )
            }

        except APIConnectionError as e:
            return {
                "success": False,
                "meal_plan": None,
                "error": (
                    f"Function: MealPlanGenerator._request_meal_plan | "
                    f"Error type: {type(e).__name__} | "
                    "Location: OpenAI API connection | "
                    "Suggestion: Check your internet connection and try again."
                )
            }

        except json.JSONDecodeError as e:
            return {
                "success": False,
                "meal_plan": None,
                "error": (
                    "Function: parse_json_response | "
                    f"Error type: {type(e).__name__} | "
                    f"Location: JSON line {e.lineno}, column {e.colno} (char {e.pos}) | "
                    "Suggestion: Check that the model response contains valid JSON."
                )
            }

        except ValidationError as e:
            first_error = e.errors()[0]
            location = ".".join(str(part) for part in first_error["loc"])

            return {
                "success": False,
                "meal_plan": None,
                "error": (
                    "Function: validate_meal_plan | "
                    f"Error type: {type(e).__name__} | "
                    f"Location: {location or 'MealPlan schema'} | "
                    f"Details: {first_error['msg']} | "
                    "Suggestion: Check that the plan contains 7 recipes "
                    "with 5 ingredients each."
                )
            }
        
        except Exception as e:
            return {
                "success": False,
                "meal_plan": None,
                "error": (
                    "Function: MealPlanGenerator.generate | "
                    f"Error type: {type(e).__name__} | "
                    "Location: Meal plan generation | "
                    f"Details: {str(e)} | "
                    "Suggestion: Check the input data and configuration, "
                    "then try again."
                )
            }


# Initialize generator
generator = MealPlanGenerator()

In [75]:
def format_ingredients(recipe: dict) -> str:
    """Format recipe ingredients as a comma-separated string."""

    ingredients = ", ".join([
        f"{ing['quantity']} {ing['name']}"
        for ing in recipe['ingredients']
    ])

    return ingredients

def use_default_if_empty(value: str | None, default: str) -> str:
    """Return the default if the input is empty, whitespace, or None."""
    return value if isinstance(value, str) and value.strip() else default

## 6. Email Functionality

In [77]:
def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = format_ingredients(recipe)
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)

def create_email_message(recipient_email: str, meal_plan: dict) -> MIMEMultipart:
    """Create an email message containing the meal plan."""

    msg = MIMEMultipart()
    msg["From"] = EMAIL_SENDER
    msg["To"] = recipient_email
    msg["Subject"] = "Your Weekly Meal Plan"

    body = format_meal_plan_email(meal_plan)
    msg.attach(MIMEText(body, "plain"))

    return msg

def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    try:
        msg = create_email_message(recipient_email, meal_plan)

        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}

    except smtplib.SMTPAuthenticationError as e:
        return {
            "success": False,
            "message": (
                "Function: send_meal_plan_email | "
                f"Error type: {type(e).__name__} | "
                "Location: SMTP authentication | "
                "Suggestion: Check EMAIL_SENDER and EMAIL_PASSWORD."
            )
        }
    
    except Exception as e:
        return {
            "success": False,
            "message": (
                "Function: send_meal_plan_email | "
                f"Error type: {type(e).__name__} | "
                "Location: Email creation or SMTP delivery | "
                f"Details: {str(e)} | "
                "Suggestion: Check the email data, SMTP configuration, "
                "and network connection."
            )
        }

## 7. Validation

In [78]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

## 8. Gradio Interface

In [79]:
def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = format_ingredients(recipe)
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"

def format_meal_plan_header(meal_plan: dict) -> str:
    """Format the introductory section of a meal plan."""
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    return output

def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output and store the result."""
    output = "Generating your meal plan..."
    yield output, None

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=use_default_if_empty(theme_requests, "random"),
        additional_preferences=use_default_if_empty(additional_prefs, "none")
    )

    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = format_meal_plan_header(meal_plan)
    yield output, meal_plan
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output, meal_plan
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output, meal_plan

def send_email_ui(email, meal_plan):
    """Send the previously generated meal plan via email."""

    if not email or "@" not in email:
        return "Enter a valid email address."

    if meal_plan is None:
        return "Please generate a meal plan before sending an email."

    email_result = send_meal_plan_email(email, meal_plan)
    return email_result["message"]

# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    plan_state = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display, plan_state]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, plan_state],
        outputs=[email_status]
    )

## 9. Launch Application

In [80]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7b8a55e0ad1787ee78.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Test Cases

Defines test cases for different dietary preferences without executing additional API calls.

In [68]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

### Step 3: Helper Functions – Refactoring Summary

I created two reusable helper functions to reduce duplicated logic in the application.

- `format_ingredients()` centralizes ingredient formatting for both the email and Gradio interface.
- `use_default_if_empty()` handles empty user inputs and provides default values consistently.

Both helpers were tested successfully using manually created test data without making OpenAI API calls. I also verified that the refactored recipe and email formatting functions produce the expected output.

I reviewed the remaining suggested patterns and found no additional duplicated logic that required a separate helper function at this stage.

The refactoring preserves the original functionality while improving code reusability and maintainability.

### Step 4: Modularizing Functions

In this step, I refactored several existing functions to improve code readability, separation of responsibilities, and maintainability.

I separated the meal plan generation process into smaller components for LLM communication, JSON parsing, and Pydantic validation. I also moved the API request into the existing error-handling block.

For the email functionality, I extracted the message creation logic into a separate function, keeping it independent from SMTP communication.

Finally, I extracted the meal plan header formatting from the Gradio streaming function to simplify the user interface logic.

I tested the new helper functions using manually created sample data without making additional OpenAI API calls or sending emails.

In Step 6, I addressed the duplicate meal plan generation issue by using Gradio session state to reuse the previously generated plan when sending an email. A mocked test confirmed that no additional generation call is made.

### Step 5: Error Handling

In this step, I tested how the original application responds to different errors and improved the error handling to provide more useful feedback.

**Initial error testing**

| Test case | Original behavior |
|---|---|
| Invalid OpenAI API key | Returned "Invalid API key" without additional context |
| Network connection failure | Returned "Connection error." without a suggestion |
| Invalid JSON | Raised a JSONDecodeError with the error position |
| Invalid meal-plan schema | Raised a Pydantic ValidationError identifying the affected field |
| Invalid email password | Returned an SMTP authentication error without clear guidance |
| Empty input / None | Empty strings were handled, but None caused an AttributeError |

I used mocked API and SMTP errors to avoid unnecessary network requests, changing real credentials, or sending emails.

**Improvements implemented**

I added specific exception handling for OpenAI authentication and connection errors, JSON parsing errors, Pydantic validation errors, and SMTP authentication errors.

The improved error messages include the function name, error type, location, and a suggested solution. I also updated the general exception handlers and modified `use_default_if_empty()` to handle `None` values safely.

**Results and limitations**

The simulated error tests confirmed that the new specific exception handlers return structured messages instead of generic errors. The JSON error handling now includes the exact line and column, while Pydantic validation errors identify the affected field.

The application handles the tested error scenarios more clearly. However, these tests do not replace full end-to-end testing with real network and email connections.

### Step 6: Error Handling Regression Tests

I compared the original error behavior with the refactored implementation using the test cases from Step 5.

| Test case | How | What does the original show? | What does mine show? |
|---|---|---|---|
| OpenAI API key | Simulated authentication failure | `Invalid API key` | Structured `AuthenticationError` with function, location, and suggestion |
| Network | Simulated connection failure | `Connection error.` | Structured `APIConnectionError` with troubleshooting suggestion |
| Invalid JSON | Passed malformed JSON | `JSONDecodeError` with traceback | Structured error with JSON line 1, column 14, and suggestion |
| Invalid meal-plan schema | Provided 6 instead of 7 recipes | Pydantic `ValidationError` with traceback | Structured `ValidationError` identifying the `recipes` field |
| Email authentication | Simulated incorrect SMTP password | Generic email failure message | Structured `SMTPAuthenticationError` with credential-check suggestion |
| Empty input / None | Passed `None` instead of a string | `AttributeError` | Returns the default value `"random"` |

**Result:** All six error scenarios have been checked. The refactored implementation provides clearer error information or safely handles the invalid input.

The API and SMTP failures were simulated without external requests. Full integration behavior still needs to be verified separately.

### Real API Test and Functional Verification

I performed one real OpenAI API test to verify that the refactored application still produces the same output structure as the original version.

The generation was successful and returned 7 recipes with exactly 5 ingredients each, along with a consolidated shopping list containing 33 items. The generation took 18.23 seconds.

I also investigated the duplicate generation issue identified earlier. I updated the Gradio interface to store the generated meal plan using `gr.State` and reuse it for email delivery. A mocked test confirmed that the email function receives the existing meal plan without triggering another generation call. The complete email workflow was not tested with a live SMTP connection.

File-based error tests were not implemented because this application accepts user input through Gradio forms rather than external files. Invalid JSON and schema validation were already tested independently without additional API calls.